# Chapter 49 — Fixed-Length Context Windows

## Learning goals

Chapter 47 predicted from one current token, and Chapter 48 converted token IDs into vectors.

This chapter builds examples containing several consecutive token IDs.

By the end of this chapter, you should be able to:

1. Define a context window and context length.
2. Build sliding input-target examples from token IDs.
3. Create one-target input batches with shape `[batch_size, context_length]`.
4. Create one-target targets with shape `[batch_size]`.
5. Decode batch rows to verify alignment.
6. Select deterministic mini-batches from a full example tensor.
7. Build GPT-style shifted targets with shape `[batch_size, context_length]`.
8. Distinguish one-target-per-context from one-target-per-position training.
9. Explain why shifted targets still require causal model computation.
10. Convert a context batch into token embeddings for later sequence processing.

## The big idea

A fixed-length context window contains a chosen number of consecutive tokens.

With context length four:

```text
context: h e l l
target:              o
```

Sliding the window one token at a time creates more examples.

Rectangular windows make it straightforward to process many examples together in tensors.

## Terms used in this chapter

- A **context** is the token information available for a prediction.
- A **context window** is a bounded slice of tokens supplied to a model.
- The **context length** is the number of token positions in a full window.
- A **target token** is the correct next token for a prediction.
- A **training example** pairs model input with its target.
- A **batch** groups examples along a leading dimension.
- A **mini-batch** is a selected subset of available training examples.
- A **sliding window** moves a fixed-size slice across a sequence.
- **One-target-per-context** training predicts one token after the complete window.
- **One-target-per-position** training predicts a next token at every sequence position.
- A **causal model** prevents a position from using future positions when predicting its target.

## Import PyTorch and stay on CPU

The examples use integer token tensors and deterministic mini-batch sampling.

No CUDA code is required.

In [1]:
import torch

device = "cpu"
RANDOM_SEED = 49

print("PyTorch version:", torch.__version__)
print("Course device:", device)
print("Random seed:", RANDOM_SEED)

assert device == "cpu"

PyTorch version: 2.2.2
Course device: cpu
Random seed: 49


## Start with a word-level example

Split one sentence on spaces and assign deterministic IDs from a sorted vocabulary.

Use context length two so every example remains easy to read.

In [2]:
word_text = "the cat sat on the mat"
word_tokens = word_text.split()
word_vocabulary = sorted(set(word_tokens))

word_to_id = {word: token_id for token_id, word in enumerate(word_vocabulary)}
id_to_word = {token_id: word for word, token_id in word_to_id.items()}
word_token_ids = [word_to_id[word] for word in word_tokens]

print("Text:", word_text)
print("Tokens:", word_tokens)
print("Vocabulary:", word_vocabulary)
print("Word to ID:", word_to_id)
print("Encoded tokens:", word_token_ids)

assert word_tokens == ["the", "cat", "sat", "on", "the", "mat"]
assert len(word_token_ids) == 6

Text: the cat sat on the mat
Tokens: ['the', 'cat', 'sat', 'on', 'the', 'mat']
Vocabulary: ['cat', 'mat', 'on', 'sat', 'the']
Word to ID: {'cat': 0, 'mat': 1, 'on': 2, 'sat': 3, 'the': 4}
Encoded tokens: [4, 0, 3, 2, 4, 1]


The IDs preserve sequence order even though the vocabulary itself is sorted.

The first context `the cat` should predict `sat`.

## Build one-target context examples

For a sequence of length `N` and full context length `C`, sliding by one position creates `N - C` examples.

Validate that at least one complete context and following target exist.

In [3]:
def build_one_target_context_examples(
    token_ids: list[int],
    context_length: int,
) -> tuple[list[list[int]], list[int]]:
    if context_length < 1:
        raise ValueError("context_length must be at least 1.")

    if len(token_ids) <= context_length:
        raise ValueError("token_ids must contain a full context and target.")

    input_examples: list[list[int]] = []
    target_examples: list[int] = []

    for start_index in range(len(token_ids) - context_length):
        context_end = start_index + context_length
        input_examples.append(token_ids[start_index:context_end])
        target_examples.append(token_ids[context_end])

    return input_examples, target_examples


word_context_length = 2
word_input_examples, word_target_examples = build_one_target_context_examples(
    token_ids=word_token_ids,
    context_length=word_context_length,
)

print("Input examples:", word_input_examples)
print("Target examples:", word_target_examples)

assert len(word_input_examples) == len(word_tokens) - word_context_length
assert all(len(example) == word_context_length for example in word_input_examples)

Input examples: [[4, 0], [0, 3], [3, 2], [2, 4]]
Target examples: [3, 2, 4, 1]


Every input list has two IDs, and every target is one ID immediately after its context.

The window width remains fixed while its start index changes.

## Decode the word examples

Readable text is the quickest way to detect an off-by-one error in a window builder.

In [4]:
print("start | context | target")
print("-" * 30)

for start_index, (context_ids, target_id) in enumerate(
    zip(
        word_input_examples,
        word_target_examples,
        strict=True,
    )
):
    context_words = [id_to_word[token_id] for token_id in context_ids]
    print(f"{start_index:>5} | {' '.join(context_words):>11} | {id_to_word[target_id]}")

start | context | target
------------------------------
    0 |     the cat | sat
    1 |     cat sat | on
    2 |      sat on | the
    3 |      on the | mat


The rows are:

```text
the cat → sat
cat sat → on
sat on  → the
on the  → mat
```

Each adjacent pair of rows overlaps by one token.

## Make the sliding motion explicit

Print the start and end indexes beside the decoded window.

Python's slice end is exclusive, and the target sits exactly at that end index.

In [5]:
print("start:end | context | target index | target")
print("-" * 51)

for start_index in range(len(word_token_ids) - word_context_length):
    end_index = start_index + word_context_length
    context_words = word_tokens[start_index:end_index]
    target_word = word_tokens[end_index]

    print(
        f"{start_index}:{end_index:<5} | "
        f"{' '.join(context_words):>11} | "
        f"{end_index:>12} | "
        f"{target_word}"
    )

start:end | context | target index | target
---------------------------------------------------
0:2     |     the cat |            2 | sat
1:3     |     cat sat |            3 | on
2:4     |      sat on |            4 | the
3:5     |      on the |            5 | mat


This one-position stride is the sliding-window rule used throughout the chapter.

Other data loaders may choose random starts or larger strides, but the alignment principle is the same.

## Move to character-level text

Use `hello hello` so spaces are visible tokens and several four-character windows exist.

Build one shared character vocabulary for every later comparison.

In [6]:
character_text = "hello hello"
character_vocabulary = sorted(set(character_text))
character_to_id = {
    character: token_id for token_id, character in enumerate(character_vocabulary)
}
id_to_character = {
    token_id: character for character, token_id in character_to_id.items()
}
character_token_ids = [character_to_id[character] for character in character_text]

print("Text:", repr(character_text))
print("Vocabulary:", character_vocabulary)
print("Character to ID:", character_to_id)
print("Encoded text:", character_token_ids)

assert len(character_text) == 11
assert " " in character_vocabulary

Text: 'hello hello'
Vocabulary: [' ', 'e', 'h', 'l', 'o']
Character to ID: {' ': 0, 'e': 1, 'h': 2, 'l': 3, 'o': 4}
Encoded text: [2, 1, 3, 3, 4, 0, 2, 1, 3, 3, 4]


## Build four-character contexts

Each complete window predicts the single character immediately after all four context characters.

In [7]:
context_length = 4
character_input_examples, character_target_examples = build_one_target_context_examples(
    token_ids=character_token_ids,
    context_length=context_length,
)

print("Number of examples:", len(character_input_examples))
print("Expected number:", len(character_text) - context_length)

assert len(character_input_examples) == 7
assert len(character_target_examples) == 7

Number of examples: 7
Expected number: 7


The first example is `hell → o`.

Later windows include the space character rather than silently discarding it.

## Decode character examples

Define one reusable decoder and print `repr` so spaces remain visible.

In [8]:
def decode_character_ids(
    token_ids: list[int],
    id_to_character: dict[int, str],
) -> str:
    return "".join(id_to_character[token_id] for token_id in token_ids)


print("row | context | target")
print("-" * 27)

for row_index, (context_ids, target_id) in enumerate(
    zip(
        character_input_examples,
        character_target_examples,
        strict=True,
    )
):
    decoded_context = decode_character_ids(
        context_ids,
        id_to_character,
    )
    print(
        f"{row_index:>3} | "
        f"{repr(decoded_context):>9} | "
        f"{repr(id_to_character[target_id])}"
    )

assert (
    decode_character_ids(
        character_input_examples[0],
        id_to_character,
    )
    == "hell"
)
assert id_to_character[character_target_examples[0]] == "o"

row | context | target
---------------------------
  0 |    'hell' | 'o'
  1 |    'ello' | ' '
  2 |    'llo ' | 'h'
  3 |    'lo h' | 'e'
  4 |    'o he' | 'l'
  5 |    ' hel' | 'l'
  6 |    'hell' | 'o'


## Convert the full example set to tensors

One-target inputs form a two-dimensional integer tensor.

One scalar target ID corresponds to each input row.

In [9]:
one_target_input_batch = torch.tensor(
    character_input_examples,
    dtype=torch.long,
    device=device,
)
one_target_target_batch = torch.tensor(
    character_target_examples,
    dtype=torch.long,
    device=device,
)

print("Input batch:")
print(one_target_input_batch)
print("Input shape:", one_target_input_batch.shape)
print("Target batch:", one_target_target_batch)
print("Target shape:", one_target_target_batch.shape)
print("Dtypes:", one_target_input_batch.dtype, one_target_target_batch.dtype)

assert one_target_input_batch.shape == torch.Size([7, 4])
assert one_target_target_batch.shape == torch.Size([7])
assert one_target_input_batch.dtype == torch.long
assert one_target_target_batch.dtype == torch.long

Input batch:
tensor([[2, 1, 3, 3],
        [1, 3, 3, 4],
        [3, 3, 4, 0],
        [3, 4, 0, 2],
        [4, 0, 2, 1],
        [0, 2, 1, 3],
        [2, 1, 3, 3]])
Input shape: torch.Size([7, 4])
Target batch: tensor([4, 0, 2, 1, 3, 3, 4])
Target shape: torch.Size([7])
Dtypes: torch.int64 torch.int64


The shape meanings are:

- `[7, 4]`: seven examples with four context IDs each.
- `[7]`: seven next-token class IDs, one per example.

## Decode rows from tensors

Tensor construction can preserve shapes while still containing incorrect alignment.

Decode the actual stored rows rather than trusting only the original Python lists.

In [10]:
print("row | input IDs | decoded context | target ID | target")
print("-" * 65)

for row_index in range(one_target_input_batch.shape[0]):
    context_ids = [
        int(token_id) for token_id in one_target_input_batch[row_index].tolist()
    ]
    target_id = int(one_target_target_batch[row_index].item())

    print(
        f"{row_index:>3} | "
        f"{str(context_ids):>13} | "
        f"{repr(decode_character_ids(context_ids, id_to_character)):>15} | "
        f"{target_id:>9} | "
        f"{repr(id_to_character[target_id])}"
    )

row | input IDs | decoded context | target ID | target
-----------------------------------------------------------------
  0 |  [2, 1, 3, 3] |          'hell' |         4 | 'o'
  1 |  [1, 3, 3, 4] |          'ello' |         0 | ' '
  2 |  [3, 3, 4, 0] |          'llo ' |         2 | 'h'
  3 |  [3, 4, 0, 2] |          'lo h' |         1 | 'e'
  4 |  [4, 0, 2, 1] |          'o he' |         3 | 'l'
  5 |  [0, 2, 1, 3] |          ' hel' |         3 | 'l'
  6 |  [2, 1, 3, 3] |          'hell' |         4 | 'o'


Decoded tensor rows match the intended sliding examples.

This inspection is a useful dataset-building habit because shape checks alone cannot catch a one-position shift mistake.

## Select a deterministic mini-batch

Indexing the full example tensors with row IDs creates a smaller batch.

Batch size and context length remain separate concepts.

In [11]:
selected_row_indexes = torch.tensor(
    [0, 2, 5],
    dtype=torch.long,
    device=device,
)
mini_input_batch = one_target_input_batch[selected_row_indexes]
mini_target_batch = one_target_target_batch[selected_row_indexes]

print("Selected rows:", selected_row_indexes)
print("Mini input batch:")
print(mini_input_batch)
print("Mini input shape:", mini_input_batch.shape)
print("Mini targets:", mini_target_batch)
print("Mini target shape:", mini_target_batch.shape)

assert mini_input_batch.shape == torch.Size([3, context_length])
assert mini_target_batch.shape == torch.Size([3])

Selected rows: tensor([0, 2, 5])
Mini input batch:
tensor([[2, 1, 3, 3],
        [3, 3, 4, 0],
        [0, 2, 1, 3]])
Mini input shape: torch.Size([3, 4])
Mini targets: tensor([4, 2, 3])
Mini target shape: torch.Size([3])


The mini-batch contains three examples, while every example still contains four context tokens.

Changing batch size does not change how much context each example provides.

## Sample row indexes reproducibly

A dedicated CPU generator makes random row selection deterministic without resetting unrelated random state.

In [12]:
mini_batch_size = 4
row_generator = torch.Generator(device=device)
row_generator.manual_seed(RANDOM_SEED)

random_row_indexes = torch.randint(
    low=0,
    high=one_target_input_batch.shape[0],
    size=(mini_batch_size,),
    generator=row_generator,
    device=device,
)
random_input_batch = one_target_input_batch[random_row_indexes]
random_target_batch = one_target_target_batch[random_row_indexes]

print("Random row indexes:", random_row_indexes)
print("Random input shape:", random_input_batch.shape)
print("Random target shape:", random_target_batch.shape)

assert random_input_batch.shape == torch.Size([mini_batch_size, context_length])
assert random_target_batch.shape == torch.Size([mini_batch_size])

Random row indexes: tensor([2, 2, 4, 2])
Random input shape: torch.Size([4, 4])
Random target shape: torch.Size([4])


Sampling with replacement can select the same row more than once.

The sampled rows are training examples, not contiguous text positions within one longer sequence.

## Fixed length enables rectangular batches

Batched tensor operations need regular dimensions.

Requiring four IDs per row allows all examples to stack into one `[batch_size, 4]` tensor.

Variable-length examples usually require padding, packing, masking, or grouping by length.

This chapter uses only complete fixed-length windows and introduces no start or padding token.

## Boundaries omit incomplete contexts

With text `hello` and context length four, only `hell → o` has a complete context.

Earlier targets do not have four preceding characters.

The current builder omits those positions rather than inventing missing context.

Later systems can add boundary tokens or padding when training on shorter prefixes is desired.

## One target versus one target per position

The current setup uses the entire window to predict one token after it:

```text
input:  h e l l
target:         o
```

GPT-style training shifts a target sequence one position to the right:

```text
input:  h e l l
target: e l l o
```

The second row provides one target for each input position.

## Build GPT-style shifted examples

Use the same sliding starts and context length.

Each target slice begins one position after its input slice and has the same length.

In [13]:
def build_gpt_style_context_examples(
    token_ids: list[int],
    context_length: int,
) -> tuple[list[list[int]], list[list[int]]]:
    if context_length < 1:
        raise ValueError("context_length must be at least 1.")

    if len(token_ids) <= context_length:
        raise ValueError("token_ids must contain an input and shifted target.")

    input_examples: list[list[int]] = []
    target_examples: list[list[int]] = []

    for start_index in range(len(token_ids) - context_length):
        input_end = start_index + context_length
        input_examples.append(token_ids[start_index:input_end])
        target_examples.append(token_ids[start_index + 1 : input_end + 1])

    return input_examples, target_examples


gpt_input_examples, gpt_target_examples = build_gpt_style_context_examples(
    token_ids=character_token_ids,
    context_length=context_length,
)

print("First input IDs:", gpt_input_examples[0])
print("First target IDs:", gpt_target_examples[0])

assert len(gpt_input_examples) == 7
assert all(len(example) == context_length for example in gpt_target_examples)

First input IDs: [2, 1, 3, 3]
First target IDs: [1, 3, 3, 4]


The first row encodes `hell → ello`.

Every target entry is the token immediately after the input token at the same column.

## Decode GPT-style rows

Print input and target strings together to verify the one-position shift.

In [14]:
print("row | input | shifted targets")
print("-" * 36)

for row_index, (input_ids, target_ids) in enumerate(
    zip(
        gpt_input_examples,
        gpt_target_examples,
        strict=True,
    )
):
    decoded_input = decode_character_ids(
        input_ids,
        id_to_character,
    )
    decoded_targets = decode_character_ids(
        target_ids,
        id_to_character,
    )

    print(f"{row_index:>3} | {repr(decoded_input):>9} | {repr(decoded_targets)}")

assert (
    decode_character_ids(
        gpt_input_examples[0],
        id_to_character,
    )
    == "hell"
)
assert (
    decode_character_ids(
        gpt_target_examples[0],
        id_to_character,
    )
    == "ello"
)

row | input | shifted targets
------------------------------------
  0 |    'hell' | 'ello'
  1 |    'ello' | 'llo '
  2 |    'llo ' | 'lo h'
  3 |    'lo h' | 'o he'
  4 |    'o he' | ' hel'
  5 |    ' hel' | 'hell'
  6 |    'hell' | 'ello'


For the first row, the position-level tasks are:

- `h → e`
- `he → l`
- `hel → l`
- `hell → o`

Those growing prefixes describe what a causal model may use at each output position.

## Convert GPT-style rows to tensors

Inputs and targets now have identical two-dimensional shapes.

In [15]:
gpt_input_batch = torch.tensor(
    gpt_input_examples,
    dtype=torch.long,
    device=device,
)
gpt_target_batch = torch.tensor(
    gpt_target_examples,
    dtype=torch.long,
    device=device,
)

print("GPT input batch:")
print(gpt_input_batch)
print("GPT input shape:", gpt_input_batch.shape)
print("GPT target batch:")
print(gpt_target_batch)
print("GPT target shape:", gpt_target_batch.shape)

assert gpt_input_batch.shape == torch.Size([7, 4])
assert gpt_target_batch.shape == torch.Size([7, 4])
torch.testing.assert_close(
    gpt_input_batch[:, 1:],
    gpt_target_batch[:, :-1],
)

GPT input batch:
tensor([[2, 1, 3, 3],
        [1, 3, 3, 4],
        [3, 3, 4, 0],
        [3, 4, 0, 2],
        [4, 0, 2, 1],
        [0, 2, 1, 3],
        [2, 1, 3, 3]])
GPT input shape: torch.Size([7, 4])
GPT target batch:
tensor([[1, 3, 3, 4],
        [3, 3, 4, 0],
        [3, 4, 0, 2],
        [4, 0, 2, 1],
        [0, 2, 1, 3],
        [2, 1, 3, 3],
        [1, 3, 3, 4]])
GPT target shape: torch.Size([7, 4])


Within every row, the last three input IDs equal the first three target IDs.

That invariant is a compact check for correct shifting.

## Compare the two target styles

The input shape is identical, but the target shape and number of supervised predictions differ.

In [16]:
one_target_prediction_count = one_target_target_batch.numel()
gpt_prediction_count = gpt_target_batch.numel()

print("style | input shape | target shape | target entries")
print("-" * 68)
print(
    "one target | "
    f"{str(one_target_input_batch.shape):>18} | "
    f"{str(one_target_target_batch.shape):>18} | "
    f"{one_target_prediction_count:>14}"
)
print(
    "GPT style  | "
    f"{str(gpt_input_batch.shape):>18} | "
    f"{str(gpt_target_batch.shape):>18} | "
    f"{gpt_prediction_count:>14}"
)

assert one_target_prediction_count == 7
assert gpt_prediction_count == 28

style | input shape | target shape | target entries
--------------------------------------------------------------------
one target | torch.Size([7, 4]) |    torch.Size([7]) |              7
GPT style  | torch.Size([7, 4]) | torch.Size([7, 4]) |             28


One-target training supplies seven target entries for these seven windows.

The overlapping GPT-style toy rows supply 28 target entries, including duplicated prediction tasks across neighboring windows.

Real data loaders choose chunking and sampling strategies deliberately rather than always materializing every overlapping row.

## Shifted targets do not enforce causality

A target tensor says what each output should predict.

It does not prevent the model at an early position from reading later input positions.

A GPT-style model must use causal computation, such as a causal attention mask, so position `t` can depend only on positions up to `t`.

That architectural constraint comes later.

## Prepare context IDs with token embeddings

Chapter 48's embedding lookup accepts the two-dimensional context batch and appends an embedding dimension.

The target IDs remain integer class labels.

In [17]:
embedding_dimension = 6
torch.manual_seed(RANDOM_SEED)
token_embedding_table = torch.nn.Embedding(
    num_embeddings=len(character_vocabulary),
    embedding_dim=embedding_dimension,
    device=device,
)

one_target_embeddings = token_embedding_table(one_target_input_batch)
gpt_input_embeddings = token_embedding_table(gpt_input_batch)

print("One-target ID shape:", one_target_input_batch.shape)
print("One-target embedding shape:", one_target_embeddings.shape)
print("GPT ID shape:", gpt_input_batch.shape)
print("GPT embedding shape:", gpt_input_embeddings.shape)
print("One-target target shape:", one_target_target_batch.shape)
print("GPT target shape:", gpt_target_batch.shape)

assert one_target_embeddings.shape == torch.Size([7, 4, 6])
assert gpt_input_embeddings.shape == torch.Size([7, 4, 6])

One-target ID shape: torch.Size([7, 4])
One-target embedding shape: torch.Size([7, 4, 6])
GPT ID shape: torch.Size([7, 4])
GPT embedding shape: torch.Size([7, 4, 6])
One-target target shape: torch.Size([7])
GPT target shape: torch.Size([7, 4])


Both training styles begin with the same token-embedding input shape.

Their later output and loss shapes differ because one predicts once per row while the other predicts once per position.

## Expected model output shapes

For vocabulary size `V`:

- A one-target model produces logits shaped `[batch_size, V]`.
- A GPT-style model produces logits shaped `[batch_size, context_length, V]`.

Cross-entropy compares those logits with targets shaped `[batch_size]` or `[batch_size, context_length]`, usually after an implementation-specific reshape.

This chapter constructs the data; later chapters build models that produce those logits.

## Validate error cases

The builders reject zero-length contexts and sequences without room for a following target.

In [18]:
for invalid_token_ids, invalid_context_length in (
    ([1, 2, 3], 0),
    ([1, 2, 3], 3),
):
    try:
        build_one_target_context_examples(
            invalid_token_ids,
            invalid_context_length,
        )
    except ValueError as validation_error:
        print("Rejected invalid request:", validation_error)
    else:
        raise AssertionError("Invalid context request should fail.")

Rejected invalid request: context_length must be at least 1.
Rejected invalid request: token_ids must contain a full context and target.


Failing early gives a clearer message than creating empty tensors that break much later in training.

## What not to do

- Do not confuse context length with batch size.
- Do not use floating-point token IDs.
- Do not trust shapes without decoding representative rows.
- Do not put the target inside the one-target input window.
- Do not confuse `[batch_size]` targets with `[batch_size, context_length]` targets.
- Do not assume shifted targets enforce causal information flow.
- Do not assume the model can access tokens outside its context window.
- Do not silently ignore boundary behavior when choosing examples.

## Gotchas

- A sequence of length `N` yields `N - C` full sliding starts for context length `C`.
- One-target and GPT-style inputs can have the same shape while their targets differ.
- Sliding GPT-style rows duplicate some position-level prediction tasks.
- Random mini-batch sampling can select the same example more than once.
- Fixed windows omit early positions unless boundary tokens or padding are added.
- Causal masking is a model constraint, not a property of shifted target tensors.
- Embedding lookup changes `[B, C]` IDs into `[B, C, D]` vectors.

## Takeaways

A fixed-length context window gives each example a regular number of token IDs.

Sliding a length-`C` window across `N` tokens creates `N - C` complete starts.

One-target-per-context training uses:

```text
inputs:  [batch_size, context_length]
targets: [batch_size]
```

GPT-style one-target-per-position training uses:

```text
inputs:  [batch_size, context_length]
targets: [batch_size, context_length]
```

Decoding rows verifies alignment, and embedding lookup prepares `[B, C]` token IDs as `[B, C, D]` floating-point vectors.

The course will ultimately use GPT-style shifted targets with causal model computation.

## What comes next

The next chapter adds position information to the `[batch_size, context_length, embedding_dimension]` token vectors.

Token embeddings identify what appears, while positional embeddings identify where each token appears inside its context window.